In [176]:
import pandas as pd
from itertools import product
import os

In [177]:
curr_dir_path = os.getcwd()
main_dir_path = os.path.abspath(os.path.join(curr_dir_path, os.pardir))
data_dir_path = os.path.join(main_dir_path, 'data')
raw_data_dir_path = os.path.join(data_dir_path, 'raw_data')
processed_data_dir_path = os.path.join(data_dir_path, 'processed_data')
cw_dir_path = os.path.join(data_dir_path, 'cw')
emissions_data_dir_path = os.path.join(raw_data_dir_path, 'emissions_data')
iea_data_dir_path = os.path.join(raw_data_dir_path, 'IEA_climate_policy_data')

In [178]:
df_policies = pd.read_csv(os.path.join(processed_data_dir_path, 'IEA_policies_clean.csv'))

# Add policy_id column
df_policies['policy_id'] = df_policies.index

df_policies

,year,jurisdiction,title,description,status,iso3,country,topic,type,category,policy_id
0,2011,National,Law 47-09 on Energy Efficiency,"Law 47-09, or the ""Law on Energy Efficiency"" a...",In force,MAR,Morocco,Unknown,Unknown,Unknown,0
1,2017,National,Swiss Energy Strategy 2050,Swiss Energy Strategy 2050The Swiss Energy Str...,In force,CHE,Switzerland,Economy-wide,"Master Energy Plan,Framework legislation",Unknown,1
2,2010,National,A Group of Energy Efficiency Measures in Agric...,This group consists of five PAMs.1. Biomass bo...,In force,FIN,Finland,Unknown,Unknown,Unknown,2
3,1999,National,New Buses,Financial assistance to regions and municipali...,In force,ITA,Italy,Unknown,Unknown,Unknown,3
4,2000,City/Municipal,New Subway System,Athens new subway system was inaugurated in Ja...,In force,GRC,Greece,Unknown,Unknown,Unknown,4
...,...,...,...,...,...,...,...,...,...,...,...
9267,2004,National,Investment Grants for SMEs and non-industrial ...,"On 30th June 2004, a law providing wide-rangi...",In force,LUX,Luxembourg,Unknown,Unknown,Unknown,9267
9268,2006,National,Energy Review,"Launched in January 2006, the Energy Review ex...",In force,GBR,United Kingdom,Unknown,Unknown,Unknown,9268
9269,1996,National,Voluntary Energy Audits,The Grand Ducal regulation of 11 August 1996 o...,In force,LUX,Luxembourg,Unknown,Unknown,Unknown,9269
9270,2006,National,Public Transit Capital Trust,Canada's Federal Budget 2006 dedicated CAD 1.3...,In force,CAN,Canada,Unknown,Unknown,Unknown,9270


In [179]:
df_policies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9272 entries, 0 to 9271
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   year          9272 non-null   int64 
 1   jurisdiction  9272 non-null   object
 2   title         9272 non-null   object
 3   description   9272 non-null   object
 4   status        9272 non-null   object
 5   iso3          9272 non-null   object
 6   country       9272 non-null   object
 7   topic         9272 non-null   object
 8   type          9272 non-null   object
 9   category      9272 non-null   object
 10  policy_id     9272 non-null   int64 
dtypes: int64(2), object(9)
memory usage: 796.9+ KB


In [180]:
df_emissions = pd.read_csv(os.path.join(processed_data_dir_path, 'total_emissions_db.csv'))
df_emissions

,Code,Year,Total Emissions
0,ABW,2000,0.335765
1,ABW,2001,0.344135
2,ABW,2002,0.363222
3,ABW,2003,0.412246
4,ABW,2004,0.430187
...,...,...,...
5285,ZWE,2018,47.509033
5286,ZWE,2019,46.442562
5287,ZWE,2020,44.576343
5288,ZWE,2021,45.759664


## Helper Functions to create Features

In [181]:
def get_number_of_policies(df):
    """
    Function to get the number of policies per year, per country
    """
    df = df.groupby(['year', 'iso3']).count().reset_index()
    df = df[['year', 'iso3', 'policy_id']]
    df = df.rename(columns={'policy_id': 'policies_per_year'})
    
    return df
  

In [182]:
def get_number_of_policies_in_the_last_years(df, n):
    """
    Function to get the number of policies in the last "n" years for each country.
    
    This function computes a rolling sum of the 'policies_per_year' for each country (grouped by 'iso3'),
    where the window is n years (including the current year). For the earliest years, where fewer than n records
    are available, the resulting sum will be NaN.
    
    Parameters:
        df (pd.DataFrame): DataFrame with columns 'iso3', 'year', and 'policies_per_year'.
        n (int): Number of years to aggregate over.
    
    Returns:
        pd.DataFrame: DataFrame with an additional column 'policies_last_n_years' that contains the computed rolling sum.
    """
    # Ensure the DataFrame is sorted by iso3 and year
    df_sorted = df.sort_values(['iso3', 'year']).copy()

    new_col_name = f'policies_last_{n}_years'
    
    # For each country (grouped by iso3), compute the rolling sum over the last n years,
    # using min_periods=n to ensure that if there are fewer than n years, the result is NaN.
    df_sorted[new_col_name] = (
        df_sorted.groupby('iso3')['policies_per_year']
                 .rolling(window=n, min_periods=n)
                 .sum()
                 .reset_index(level=0, drop=True)
    )

    # Make sure policies_last_n_years is an integer
    df_sorted[new_col_name] = df_sorted[new_col_name].fillna(0).astype(int)
    
    return df_sorted


In [183]:
def get_previous_year_emissions(df):
    """
    Function to get the emissions of the previous year for each year in each country.
    For the earliest available year for a country, the previous year emission is set 
    to the current year's emission value.

    Parameters:
        df (pd.DataFrame): DataFrame with columns 'Code', 'Year', and 'Total Emissions'.
    
    Returns:
        pd.DataFrame: DataFrame with a new column 'prev_year_emission' containing the emission 
                      of the previous year for each country-year. For the earliest year, it uses 
                      the current year's emission.
    """
    # Sort by country code and year to ensure correct chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()
    
    # Compute previous year emissions using shift within each country group
    df_sorted['prev_year_emission'] = (
        df_sorted.groupby('Code')['Total Emissions']
                 .shift(1)
    )
    
    # For rows where previous year emissions is NaN (i.e., the earliest year for each country),
    # fill with the current year's emission value.
    mask = df_sorted['prev_year_emission'].isna()
    df_sorted.loc[mask, 'prev_year_emission'] = df_sorted.loc[mask, 'Total Emissions']
    
    return df_sorted


In [184]:
def get_avg_emissions_in_previous_years(df, n):
    """
    Function to get the average emissions in the previous "n" years for each country in each year.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing columns 'Code', 'Year', and 'Total Emissions'.
        n (int): Number of years to average over (using the previous n years, not including the current year).
    
    Returns:
        pd.DataFrame: A DataFrame with an additional column 'avg_emissions_prev_n_years' that contains the average 
                      emissions for the previous n years for each country-year. For the earliest year(s) where previous 
                      data is not available, the value will be NaN.
    """
    # Sort the DataFrame by 'Code' and 'Year' to ensure proper chronological order
    df_sorted = df.sort_values(['Code', 'Year']).copy()

    # new col name
    new_col_name = f'avg_emissions_prev_{n}_years'

    # Use transform so the computed series aligns with the original DataFrame's index.
    df_sorted[new_col_name] = df_sorted.groupby('Code')['Total Emissions'].transform(
        lambda x: x.shift(1).rolling(window=n, min_periods=1).mean()
    )

    
    # For rows where avg_emissions_prev_n_years is NaN (i.e., the earliest year for each country),
    # fill with the current year's emission value.
    mask = df_sorted[new_col_name].isna()
    df_sorted.loc[mask, new_col_name] = df_sorted.loc[mask, 'Total Emissions']
    
    return df_sorted



## Create Dataset for Prediction Model

In [185]:
df_policies_per_year = get_number_of_policies(df_policies)
df_policies_per_year

,year,iso3,policies_per_year
0,1948,IND,1
1,1951,NOR,1
2,1967,AUT,1
3,1969,NGA,1
4,1970,ARG,1
...,...,...,...
2885,2025,SWE,1
2886,2025,UKR,1
2887,2026,CAN,1
2888,2027,USA,2


In [186]:
unique_isos = df_policies['iso3'].unique()

max_year = df_policies['year'].max()
min_year = df_policies['year'].min()
unique_years = range(min_year, max_year + 1)

# Create a df with all possible combinations of iso3 and year
combinations = list(product(unique_isos, unique_years))
df_combinations = pd.DataFrame(combinations, columns=['iso3', 'year'])

# Sort by iso3 and year
df_combinations = df_combinations.sort_values(by=['iso3', 'year']).reset_index(drop=True)
df_combinations.head()

,iso3,year
0,AFG,1948
1,AFG,1949
2,AFG,1950
3,AFG,1951
4,AFG,1952


In [187]:
df_template = pd.merge(df_combinations, df_policies_per_year, on=['iso3', 'year'], how='left')
df_template = df_template.fillna(0)

# Make policies per year int
df_template['policies_per_year'] = df_template['policies_per_year'].astype(int)
df_template

,iso3,year,policies_per_year
0,AFG,1948,0
1,AFG,1949,0
2,AFG,1950,0
3,AFG,1951,0
4,AFG,1952,0
...,...,...,...
17491,ZWE,2024,0
17492,ZWE,2025,0
17493,ZWE,2026,0
17494,ZWE,2027,0


In [188]:
policies_in_the_last_years= get_number_of_policies_in_the_last_years(df_template, 3)
policies_in_the_last_years.head()

,iso3,year,policies_per_year,policies_last_3_years
0,AFG,1948,0,0
1,AFG,1949,0,0
2,AFG,1950,0,0
3,AFG,1951,0,0
4,AFG,1952,0,0


In [189]:
prev_year_emissions_df = get_previous_year_emissions(df_emissions)
prev_year_emissions_df.head()

,Code,Year,Total Emissions,prev_year_emission
0,ABW,2000,0.335765,0.335765
1,ABW,2001,0.344135,0.335765
2,ABW,2002,0.363222,0.344135
3,ABW,2003,0.412246,0.363222
4,ABW,2004,0.430187,0.412246


In [190]:
avg_emissions_prev_years_df = get_avg_emissions_in_previous_years(prev_year_emissions_df, 3)
avg_emissions_prev_years_df.head()

,Code,Year,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
0,ABW,2000,0.335765,0.335765,0.335765
1,ABW,2001,0.344135,0.335765,0.335765
2,ABW,2002,0.363222,0.344135,0.339950
3,ABW,2003,0.412246,0.363222,0.347708
4,ABW,2004,0.430187,0.412246,0.373201


In [191]:
avg_emissions_prev_years_df[avg_emissions_prev_years_df['Code'] == 'MEX'].head()

,Code,Year,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
2967,MEX,2000,426.618456,426.618456,426.618456
2968,MEX,2001,431.652043,426.618456,426.618456
2969,MEX,2002,453.963007,431.652043,429.135249
2970,MEX,2003,479.173415,453.963007,437.411169
2971,MEX,2004,487.731101,479.173415,454.929488


In [192]:
# Merge all the DataFrames
df_final = pd.merge(policies_in_the_last_years, avg_emissions_prev_years_df, left_on=['iso3', 'year'], right_on=['Code', 'Year'], how='inner')

# Drop Code and Year col
df_final = df_final.drop(columns=['Code', 'Year'])

df_final

,iso3,year,policies_per_year,policies_last_3_years,Total Emissions,prev_year_emission,avg_emissions_prev_3_years
0,AFG,2000,0,0,25.390391,25.390391,25.390391
1,AFG,2001,0,0,23.723115,25.390391,25.390391
2,AFG,2002,0,0,26.383509,23.723115,24.556753
3,AFG,2003,0,0,27.071538,26.383509,25.165672
4,AFG,2004,0,0,27.128799,27.071538,25.726054
...,...,...,...,...,...,...,...
4825,ZWE,2018,0,0,47.509033,45.410983,46.390136
4826,ZWE,2019,1,1,46.442562,47.509033,46.277579
4827,ZWE,2020,0,1,44.576343,46.442562,46.454193
4828,ZWE,2021,1,2,45.759664,44.576343,46.175979


In [193]:
# Make all cols lowercase
df_final.columns = df_final.columns.str.lower()

# Rename Total emissions to total_emissions and locate it at the end
df_final = df_final.rename(columns={'total emissions': 'total_emissions'})
cols = df_final.columns.tolist()
cols.remove('total_emissions')
cols.append('total_emissions')
df_final = df_final[cols]
df_final

,iso3,year,policies_per_year,policies_last_3_years,prev_year_emission,avg_emissions_prev_3_years,total_emissions
0,AFG,2000,0,0,25.390391,25.390391,25.390391
1,AFG,2001,0,0,25.390391,25.390391,23.723115
2,AFG,2002,0,0,23.723115,24.556753,26.383509
3,AFG,2003,0,0,26.383509,25.165672,27.071538
4,AFG,2004,0,0,27.071538,25.726054,27.128799
...,...,...,...,...,...,...,...
4825,ZWE,2018,0,0,45.410983,46.390136,47.509033
4826,ZWE,2019,1,1,47.509033,46.277579,46.442562
4827,ZWE,2020,0,1,46.442562,46.454193,44.576343
4828,ZWE,2021,1,2,44.576343,46.175979,45.759664


In [194]:
# Save the final DataFrame
df_final.to_csv(os.path.join(processed_data_dir_path, 'policies_ml_db.csv'), index=False)